# Airport Selection and Analysis

Choosing the right base of operations is a critical logistics decision for airborne
campaigns. The airport must have adequate runway length and surface for the aircraft,
be close enough to the study area to maximize on-station time, and have the
infrastructure to support flight operations. This notebook demonstrates HyPlan's
`airports` module for finding, filtering, and analyzing airports.

We cover:

1. Initializing the airport database with filters (country, runway, surface type)
2. Finding the nearest airport to a study site
3. Finding multiple nearby airports
4. Searching within a radius
5. Inspecting airport details with the `Airport` class
6. Analyzing runway information
7. Comparing candidate airports
8. Exporting airports to GeoJSON
9. Visualizing airports on a map

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from hyplan.airports import (
    Airport,
    initialize_data,
    find_nearest_airport,
    find_nearest_airports,
    airports_within_radius,
    get_airports,
    get_runways,
    get_airport_details,
    get_longest_runway,
    get_runway_details,
    generate_geojson,
)

## 1. Initializing the Airport Database

`initialize_data()` downloads and caches airport/runway data from the OurAirports
database. You can filter by:
- **countries** — ISO country codes (e.g., "US", "CA")
- **min_runway_length** — Minimum runway length in feet
- **runway_surface** — Accepted surface types (e.g., asphalt, concrete)
- **airport_types** — Airport categories (e.g., "large_airport", "medium_airport")

In [ ]:
# Initialize with filters suitable for a King Air B200
# (needs ~5000 ft paved runway)
initialize_data(
    countries=["US"],
    min_runway_length=5000,  # feet
    runway_surface=["ASPH", "CONC", "ASP", "CON", "MAC", "BIT", "Asphalt", "Concrete"],
)

# Check how many airports match
gdf_airports = get_airports()
print(f"Airports matching filters: {len(gdf_airports)}")

## 2. Finding the Nearest Airport

`find_nearest_airport()` returns the ICAO code of the closest airport to a given
latitude/longitude. This is a quick way to identify the most convenient base.

In [ ]:
# Study site: Central Valley, California
study_lat, study_lon = 36.5, -119.5

nearest = find_nearest_airport(study_lat, study_lon)
print(f"Nearest airport to ({study_lat}, {study_lon}): {nearest}")

# Get details
apt = Airport(nearest)
print(f"  Name: {apt.name}")
print(f"  IATA: {apt.iata_code}")
print(f"  Location: ({apt.latitude:.4f}, {apt.longitude:.4f})")
print(f"  Elevation: {apt.elevation}")
print(f"  Municipality: {apt.municipality}")

## 3. Finding Multiple Nearby Airports

`find_nearest_airports()` returns the N closest airports, giving you a ranked
list of candidates to evaluate.

In [ ]:
# Find the 5 nearest airports
top5 = find_nearest_airports(study_lat, study_lon, n=5)
print(f"Top 5 nearest airports to ({study_lat}, {study_lon}):")
for icao in top5:
    a = Airport(icao)
    print(f"  {icao} — {a.name} ({a.municipality})")

## 4. Searching Within a Radius

`airports_within_radius()` finds all airports within a specified distance.
Use `return_details=True` to get a full GeoDataFrame with airport metadata.

In [ ]:
# ICAO codes only
within_50mi = airports_within_radius(study_lat, study_lon, radius=50, unit="miles")
print(f"Airports within 50 miles: {len(within_50mi)}")
print(f"  ICAO codes: {within_50mi}")

# With full details as a GeoDataFrame
within_100km = airports_within_radius(
    study_lat, study_lon, radius=100, unit="kilometers", return_details=True
)
print(f"\nAirports within 100 km: {len(within_100km)}")
within_100km

## 5. The Airport Class

The `Airport` class provides a convenient object-oriented interface for inspecting
a single airport's properties: name, codes, location, elevation, and runway data.

In [ ]:
# Inspect a well-known airport
lax = Airport("KLAX")

print(f"Name:         {lax.name}")
print(f"ICAO:         {lax.icao_code}")
print(f"IATA:         {lax.iata_code}")
print(f"Country:      {lax.country}")
print(f"Municipality: {lax.municipality}")
print(f"Location:     ({lax.latitude:.4f}, {lax.longitude:.4f})")
print(f"Elevation:    {lax.elevation}")
print(f"Elevation ft: {lax.elevation_ft} ft")
print(f"Geometry:     {lax.geometry}")
print(f"\nRunways:")
lax.runways

## 6. Runway Analysis

Use `get_runway_details()` and `get_longest_runway()` to evaluate whether
candidate airports can accommodate your aircraft.

In [ ]:
# Runway details for several candidate airports
candidates = ["KLAX", "KBFL", "KFAT", "KVIS"]
runway_df = get_runway_details(candidates)
print(f"Runway details for {len(candidates)} airports:")
runway_df

In [ ]:
# Check longest runway at each candidate
print("Longest runway at each airport:")
for icao in candidates:
    longest = get_longest_runway(icao)
    a = Airport(icao)
    print(f"  {icao} ({a.name}): {longest:.0f} ft")

## 7. Comparing Candidate Airports

Build a comparison table to evaluate airports on multiple criteria: distance to
study site, runway length, and elevation.

In [ ]:
from hyplan.geometry import haversine

# Compare the top 5 nearest airports
rows = []
for icao in top5:
    a = Airport(icao)
    dist_km = haversine(study_lat, study_lon, a.latitude, a.longitude) / 1000
    longest_rwy = get_longest_runway(icao)
    rows.append({
        "ICAO": icao,
        "Name": a.name,
        "Municipality": a.municipality,
        "Distance (km)": f"{dist_km:.1f}",
        "Longest Runway (ft)": f"{longest_rwy:.0f}",
        "Elevation (ft)": f"{a.elevation_ft:.0f}" if a.elevation_ft else "N/A",
    })

comparison = pd.DataFrame(rows)
comparison

## 8. Batch Airport Details

`get_airport_details()` retrieves metadata for multiple airports at once, and
`get_airports()` returns the full filtered airport GeoDataFrame.

In [ ]:
# Batch details
details = get_airport_details(top5)
details

## 9. Exporting to GeoJSON

`generate_geojson()` creates a GeoJSON file of airports for use in GIS tools
or interactive maps. You can export all airports or a specific subset.

In [ ]:
# Export candidate airports to GeoJSON (commented to avoid writing files)
# generate_geojson("candidate_airports.geojson", icao_codes=top5)
# generate_geojson("all_filtered_airports.geojson")
print("Export examples:")
print('  generate_geojson("candidate_airports.geojson", icao_codes=top5)')
print('  generate_geojson("all_filtered_airports.geojson")')

## 10. Visualizing Airports on a Map

Plot candidate airports relative to the study site to assess geographic
convenience and identify potential refueling stops.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

# Plot all filtered airports in the region
gdf_region = gdf_airports[
    (gdf_airports["latitude"].between(34, 38)) &
    (gdf_airports["longitude"].between(-122, -117))
]
ax.scatter(gdf_region["longitude"], gdf_region["latitude"],
           c="lightblue", s=20, alpha=0.5, edgecolors="gray", label="All airports")

# Highlight candidates
for icao in top5:
    a = Airport(icao)
    ax.scatter(a.longitude, a.latitude, c="blue", s=80, edgecolors="black", zorder=5)
    ax.annotate(f" {icao}", (a.longitude, a.latitude), fontsize=8, fontweight="bold")

# Study site
ax.scatter(study_lon, study_lat, c="red", s=150, marker="*", edgecolors="black",
           zorder=10, label="Study site")

# 50-mile radius circle (approximate)
circle_angles = np.linspace(0, 2 * np.pi, 100)
r_deg = 50 * 1.60934 / 111.0  # approximate conversion
ax.plot(study_lon + r_deg * np.cos(circle_angles),
        study_lat + r_deg * np.sin(circle_angles),
        "r--", alpha=0.4, label="~50 mile radius")

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Airport Candidates — Central Valley, CA")
ax.legend(loc="upper right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 11. Re-initializing with Different Filters

You can call `initialize_data()` again with different criteria. For example,
when planning for a larger aircraft like the NASA ER-2, you need longer runways.

In [ ]:
# Re-initialize for NASA ER-2 (needs 8000+ ft paved runway)
initialize_data(
    countries=["US"],
    min_runway_length=8000,
    runway_surface=["ASPH", "CONC", "ASP", "CON", "Asphalt", "Concrete"],
)

gdf_er2 = get_airports()
print(f"Airports suitable for ER-2: {len(gdf_er2)}")

# Find candidates near the same study site
er2_candidates = airports_within_radius(study_lat, study_lon, radius=150, unit="miles")
print(f"\nER-2 capable airports within 150 miles:")
for icao in er2_candidates:
    a = Airport(icao)
    dist = haversine(study_lat, study_lon, a.latitude, a.longitude) / 1000
    rwy = get_longest_runway(icao)
    print(f"  {icao} — {a.name}: {rwy:.0f} ft runway, {dist:.0f} km away")

## Summary

| Function/Class | Purpose |
|----------------|---------|
| `initialize_data(countries, min_runway_length, ...)` | Load and filter the airport database |
| `find_nearest_airport(lat, lon)` | ICAO code of the closest airport |
| `find_nearest_airports(lat, lon, n)` | N closest airports ranked by distance |
| `airports_within_radius(lat, lon, radius, unit)` | All airports within a distance |
| `Airport(icao)` | Object with name, location, elevation, runways |
| `get_airport_details(icao_list)` | Batch metadata lookup |
| `get_runway_details(icao_list)` | Runway specifications for multiple airports |
| `get_longest_runway(icao)` | Longest runway length in feet |
| `get_airports()` | Full filtered airport GeoDataFrame |
| `get_runways()` | Full filtered runway DataFrame |
| `generate_geojson(filepath, icao_codes)` | Export airports to GeoJSON |